# Importing libraries and 

In [2]:
import pandas as pd
from pathlib import Path

import nltk
from nltk.tokenize import TreebankWordTokenizer, word_tokenize

nltk.download("punkt")
nltk.download("punkt_tab")

ModuleNotFoundError: No module named 'nltk'

In [ ]:
PROJECT_ROOT = Path.cwd().parent
TRAIN_PATH = PROJECT_ROOT / "data" / "raw" / "train.csv"

train_df = pd.read_csv(TRAIN_PATH)

print(train_df.shape)
train_df.head()

(3668, 4)


,idx,sentence1,sentence2,label
0,0,"Amrozi accused his brother , whom he called "" ...","Referring to him as only "" the witness "" , Amr...",1
1,1,Yucaipa owned Dominick 's before selling the c...,Yucaipa bought Dominick 's in 1995 for $ 693 m...,0
2,2,They had published an advertisement on the Int...,"On June 10 , the ship 's owners had published ...",1
3,3,"Around 0335 GMT , Tab shares were up 19 cents ...","Tab shares jumped 20 cents , or 4.6 % , to set...",0
4,4,"The stock rose $ 2.11 , or about 11 percent , ...",PG & E Corp. shares jumped $ 1.63 or 8 percent...,1


In [ ]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3668 entries, 0 to 3667
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   idx        3668 non-null   int64 
 1   sentence1  3668 non-null   object
 2   sentence2  3668 non-null   object
 3   label      3668 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 114.8+ KB


In [ ]:
treebank = TreebankWordTokenizer()

def tokenize_treebank(text):
    return treebank.tokenize(text.lower())

def tokenize_nltk_word(text):
    return word_tokenize(text.lower())

With a small training set, keeping capitalization creates more distinct token types with fewer examples of each. Lowercasing reduces vocabulary sparsity, although it sacrifices some proper-noun information.

In [ ]:
sample_sentences = pd.concat([
    train_df["sentence1"],
    train_df["sentence2"]
]).sample(
    n=10,
    random_state=42
).tolist()

In [ ]:
for i, sentence in enumerate(sample_sentences, start=1):
    print("=" * 100)
    print(f"SENTENCE {i}")
    print(sentence)

    print("\nTreebank:")
    print(tokenize_treebank(sentence))

    print("\nword_tokenize:")
    print(tokenize_nltk_word(sentence))

    print()

SENTENCE 1
By 10 p.m. , Claudette was centered about 320 miles east of Brownsville , with maximum sustained winds of 65 mph , 9 mph shy of hurricane strength .

Treebank:
['by', '10', 'p.m.', ',', 'claudette', 'was', 'centered', 'about', '320', 'miles', 'east', 'of', 'brownsville', ',', 'with', 'maximum', 'sustained', 'winds', 'of', '65', 'mph', ',', '9', 'mph', 'shy', 'of', 'hurricane', 'strength', '.']

word_tokenize:
['by', '10', 'p.m.', ',', 'claudette', 'was', 'centered', 'about', '320', 'miles', 'east', 'of', 'brownsville', ',', 'with', 'maximum', 'sustained', 'winds', 'of', '65', 'mph', ',', '9', 'mph', 'shy', 'of', 'hurricane', 'strength', '.']

SENTENCE 2
Mr Eddington described Monday 's talks with union leaders as " sensible " .

Treebank:
['mr', 'eddington', 'described', 'monday', "'s", 'talks', 'with', 'union', 'leaders', 'as', '``', 'sensible', '``', '.']

word_tokenize:
['mr', 'eddington', 'described', 'monday', "'s", 'talks', 'with', 'union', 'leaders', 'as', '``', 'sens

In [ ]:
all_sentences = pd.concat([
    train_df["sentence1"],
    train_df["sentence2"]
]).tolist()

different = []

for sentence in all_sentences:
    treebank_tokens = tokenize_treebank(sentence)
    word_tokens = tokenize_nltk_word(sentence)

    if treebank_tokens != word_tokens:
        different.append({
            "sentence": sentence,
            "treebank": treebank_tokens,
            "word_tokenize": word_tokens
        })

print("Total sentences:", len(all_sentences))
print("Different tokenizations:", len(different))
print(
    f"Difference rate: {len(different) / len(all_sentences) * 100:.2f}%"
)

Total sentences: 7336
Different tokenizations: 93
Difference rate: 1.27%


In [ ]:
for i, example in enumerate(different[:15], start=1):
    print(f"\n--- DIFFERENCE {i} ---")
    print("Sentence:", example["sentence"])
    print("Treebank:     ", example["treebank"])
    print("word_tokenize:", example["word_tokenize"])


--- DIFFERENCE 1 ---
Sentence: Results from No. 2 U.S. soft drink maker PepsiCo Inc . PEP.N were likely to be in the spotlight .
Treebank:      ['results', 'from', 'no.', '2', 'u.s.', 'soft', 'drink', 'maker', 'pepsico', 'inc', '.', 'pep.n', 'were', 'likely', 'to', 'be', 'in', 'the', 'spotlight', '.']
word_tokenize: ['results', 'from', 'no', '.', '2', 'u.s.', 'soft', 'drink', 'maker', 'pepsico', 'inc', '.', 'pep.n', 'were', 'likely', 'to', 'be', 'in', 'the', 'spotlight', '.']

--- DIFFERENCE 2 ---
Sentence: Gov. Rick Perry has said that while he opposes gambling expansion , he would be reluctant to veto continuation of the Lottery Commission .
Treebank:      ['gov.', 'rick', 'perry', 'has', 'said', 'that', 'while', 'he', 'opposes', 'gambling', 'expansion', ',', 'he', 'would', 'be', 'reluctant', 'to', 'veto', 'continuation', 'of', 'the', 'lottery', 'commission', '.']
word_tokenize: ['gov', '.', 'rick', 'perry', 'has', 'said', 'that', 'while', 'he', 'opposes', 'gambling', 'expansion', '

Treebank preserves common newswire abbreviations as single tokens, which keeps semantically meaningful units intact and avoids fragmenting their counts across multiple token forms.

In [ ]:
from nltk.tokenize import TreebankWordTokenizer

tokenizer = TreebankWordTokenizer()

def tokenize(text):
    return tokenizer.tokenize(text.lower())

In [ ]:
from collections import Counter

token_counts = Counter()

for column in ["sentence1", "sentence2"]:
    for sentence in train_df[column]:
        token_counts.update(tokenize(sentence))

print("Total tokens:", sum(token_counts.values()))
print("Unique tokens:", len(token_counts))

print("\n20 most common tokens:")
print(token_counts.most_common(20))

Total tokens: 161061
Unique tokens: 13905

20 most common tokens:
[('the', 8477), (',', 7978), ('.', 7510), ('to', 3613), ('of', 3420), ('in', 3119), ('a', 3109), ('and', 2879), ('``', 2523), ('said', 1796), ("'s", 1507), ('on', 1338), ('for', 1253), ('that', 1198), ('was', 1142), ('is', 916), ('at', 904), ('$', 820), ('with', 805), ('percent', 765)]


In [ ]:
cutoffs = [1, 2, 3, 5, 10]

total_tokens = sum(token_counts.values())

for min_count in cutoffs:
    vocab = {
        token
        for token, count in token_counts.items()
        if count >= min_count
    }

    removed_types = len(token_counts) - len(vocab)

    removed_token_occurrences = sum(
        count
        for token, count in token_counts.items()
        if count < min_count
    )

    print(f"\nmin_count = {min_count}")
    print(f"Vocabulary size: {len(vocab):,}")
    print(f"Removed unique tokens: {removed_types:,}")
    print(
        f"Training token occurrences mapped to <UNK>: "
        f"{removed_token_occurrences:,} "
        f"({removed_token_occurrences / total_tokens * 100:.2f}%)"
    )


min_count = 1
Vocabulary size: 13,905
Removed unique tokens: 0
Training token occurrences mapped to <UNK>: 0 (0.00%)

min_count = 2
Vocabulary size: 9,926
Removed unique tokens: 3,979
Training token occurrences mapped to <UNK>: 3,979 (2.47%)

min_count = 3
Vocabulary size: 5,928
Removed unique tokens: 7,977
Training token occurrences mapped to <UNK>: 11,975 (7.44%)

min_count = 5
Vocabulary size: 3,753
Removed unique tokens: 10,152
Training token occurrences mapped to <UNK>: 19,565 (12.15%)

min_count = 10
Vocabulary size: 1,965
Removed unique tokens: 11,940
Training token occurrences mapped to <UNK>: 31,329 (19.45%)
